In [19]:
# Import library yang dibutuhkan
import undetected_chromedriver as uc  # Otomasi browser & bypass proteksi Cloudflare WAF
from bs4 import BeautifulSoup           # Parsing dan ekstraksi struktur DOM HTML
import time                             # Mengatur jeda (delay) render halaman
import os                               # Mengecek keberadaan file di sistem lokal
import warnings                         # Menyaring peringatan yang tidak relevan
import re                               # Regular expression untuk pembersihan tag & pencarian teks
import csv                              # Menulis data ke file CSV berformat tabel


In [20]:
# Inisialisasi Chrome driver (versi 153 sesuai Google Chrome di laptop)
# Driver dibuka satu kali di awal agar proses scraping berjalan efisien
driver = uc.Chrome(version_main=153)


In [21]:
def generateFileCSV(listHasil, csvName1):
    csvFolder = "./"
    csvName = csvName1

    # Header kolom sesuai format informasi meta perkara putusan
    header_kolom = (
        "terdakwa", "penuntut_umum", "nomor", "tingkat_proses", "klasifikasi", 
        "kata_kunci", "tahun", "tanggal_register", "lembaga_peradilan", 
        "jenis_lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", 
        "amar", "amar_lainnya", "catatan_amar", "tanggal_musyawarah", 
        "tanggal_dibacakan", "kaidah", "abstrak", "url"
    )

    # Gunakan encoding UTF-8 agar karakter khusus (tanda baca, simbol hukum) tidak error
    if os.path.exists(csvName):
        f = open(csvName, 'a', newline='', encoding='utf-8')
        print("File CSV sudah ada, menambahkan data...")
        w = csv.writer(f)
    else:
        f = open(csvName, 'w', newline='', encoding='utf-8')
        print("File CSV belum ada, membuat file baru...")
        w = csv.writer(f)
        w.writerow(header_kolom)

    # Tulis setiap baris data putusan ke file CSV
    for s in listHasil:
        w.writerow(s)

    f.close()
    return "\nCreate Csv file Berhasil\n"


In [22]:
def generateMeta(urlMeta):
    url = str(urlMeta).strip()
    
    # Akses halaman detail putusan dengan browser Chrome
    driver.get(url)
    time.sleep(3)  # Jeda 3 detik agar Cloudflare memvalidasi dan halaman selesai dirender

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    cleanTags = re.compile('<.*?>')  # Pola regex untuk membersihkan tag HTML (<...>)

    listMeta = []

    # Cari tabel meta info perkara pada tag ul class 'portfolio-meta nobottommargin'
    portfolio_meta = soup.find("ul", {"class": "portfolio-meta nobottommargin"})
    if not portfolio_meta:
        return []

    table = portfolio_meta.find("table")
    rowsMETA1 = table.findAll("tr") if table else []
    rowsMETA2 = soup.findAll("ul", {"class": "portfolio-meta nobottommargin"})

    # Ekstraksi setiap baris pada tabel metadata
    for row in rowsMETA1:
        coll = row.findAll("td")

        # Jika ada 2 kolom (Label & Nilai), ambil nilai di kolom kedua
        if len(coll) > 1:
            cleantext2 = (re.sub(cleanTags, '', str(coll[1]))).strip()
            listMeta.append(cleantext2.replace('\n', ' '))
        else:
            # Kolom tunggal biasanya berisi identitas pihak (Terdakwa / Penuntut Umum)
            cleantext1 = (re.sub(cleanTags, ' ', str(coll[0]))).strip()
            pidorpdt = re.search(r'(.*)/Pdt.(.*)', str(cleantext1), re.M | re.I)

            # Identifikasi perkara pidana atau perdata
            if pidorpdt == None:
                entTerdakwah = re.search(r'(.*)Terdakwa:(.*)', str(cleantext1), re.M | re.I)
                entPenuntut = re.search(r'Penuntut Umum:(.*)Terdakwa:', str(cleantext1), re.M | re.I)
            else:
                entTerdakwah = re.search(r'(.*)Tergugat:(.*)', str(cleantext1), re.M | re.I)
                entPenuntut = re.search(r'Penggugat:(.*)Tergugat:', str(cleantext1), re.M | re.I)

            # Ambil nama terdakwa jika pola cocok
            if entTerdakwah == None:
                listMeta.append("")
            else:
                listMeta.append(entTerdakwah.group(2).strip())

            # Ambil nama penuntut umum jika pola cocok
            if entPenuntut == None:
                listMeta.append("")
            else:
                listMeta.append(entPenuntut.group(1).strip())

    # Ekstraksi tautan unduh dokumen putusan jika ada
    if len(rowsMETA2) > 1:
        urlDL = rowsMETA2[1].findAll("li")
        if len(urlDL) > 4:
            urlDLStr = str(urlDL[4])
            listMeta.append(urlDLStr[urlDLStr.find("https"):urlDLStr.find('">')])
        else:
            listMeta.append("")
    else:
        listMeta.append("")

    return listMeta


In [23]:
def main():
    warnings.filterwarnings('ignore')

    # Konfigurasi file input dan file output CSV sesuai penamaan tugas (IE-NamaPN-nim.csv)
    fileListURL = "hasilListURLPage.txt"
    fileMetaCSV = "IE-PN_Blitar-240411100135.csv" 
    listHasil = []

    startTime = time.time()
    
    # Membaca daftar URL putusan yang sudah diambil pada tahap 1
    openfileListURL = open(fileListURL, "r", encoding='UTF8')
    rawListURL = openfileListURL.readlines()
    openfileListURL.close()

    # Filter: Hapus link duplikat dari sidebar agar setiap putusan unik dan hemat waktu
    bacaListURL = list(dict.fromkeys([u.strip() for u in rawListURL if u.strip()]))
    print(f"Total URL putusan unik yang akan diproses: {len(bacaListURL)}")

    # Iterasi pengambilan metadata dari setiap URL putusan
    i = 1
    for barisURL in bacaListURL:
        try:
            hasil = generateMeta(str(barisURL))
            if hasil:
                listHasil.append(hasil)
                print(f"======= ROW HASIL ======= {i} / {len(bacaListURL)}")
            else:
                print(f"Row {i}: Format tidak sesuai / kosong")
        except Exception as e:
            print(f"Error Get Meta Inf row {i}, {e}")
        i = i + 1

    # Simpan kumpulan metadata yang berhasil diambil ke dalam file CSV
    createFile = generateFileCSV(listHasil, fileMetaCSV)
    print(createFile)

    endTime = time.time()
    print('Semua proses scraping metadata selesai!')
    print('Total waktu pemrosesan : ', round(endTime - startTime, 2), ' Detik')

main()


Total URL putusan unik yang akan diproses: 237
======= ROW HASIL ======= 1 / 237
Row 2: Format tidak sesuai / kosong
======= ROW HASIL ======= 3 / 237
======= ROW HASIL ======= 4 / 237
======= ROW HASIL ======= 5 / 237
Row 6: Format tidak sesuai / kosong
Row 7: Format tidak sesuai / kosong
======= ROW HASIL ======= 8 / 237
Row 9: Format tidak sesuai / kosong
======= ROW HASIL ======= 10 / 237
======= ROW HASIL ======= 11 / 237
======= ROW HASIL ======= 12 / 237
======= ROW HASIL ======= 13 / 237
======= ROW HASIL ======= 14 / 237
======= ROW HASIL ======= 15 / 237
======= ROW HASIL ======= 16 / 237
======= ROW HASIL ======= 17 / 237
======= ROW HASIL ======= 18 / 237
======= ROW HASIL ======= 19 / 237
======= ROW HASIL ======= 20 / 237
======= ROW HASIL ======= 21 / 237
======= ROW HASIL ======= 22 / 237
======= ROW HASIL ======= 23 / 237
======= ROW HASIL ======= 24 / 237
======= ROW HASIL ======= 25 / 237
======= ROW HASIL ======= 26 / 237
======= ROW HASIL ======= 27 / 237
======= R

In [6]:
# Tutup sesi browser setelah proses scraping selesai
driver.quit()
